In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/
!cp "/content/drive/MyDrive/OmniVoice.zip" /content/
!unzip -o -q OmniVoice.zip

%cd OmniVoice-Colab
!pip install -e .
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Mounted at /content/drive
/content
/content/OmniVoice-Colab
Obtaining file:///content/OmniVoice-Colab
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 8.6 MB/s eta 0:00:00
  Building editable for omnivoice (pyproject.toml) ... done
  Created wheel for omnivoice: filename=omnivoice-0.1.5-py3-none-any.whl size=6796 sha256=206d935be5b6b238ce209e0aff83b03591a9c7d9e4c1a7a134978e446210eb61
  Stored in directory: /tmp/pip-ephem-wheel-cache-6voddkjq/wheels/8d/d2/2f/7bfb816ba1e13b1fd90a521e86a2b22d73d0c7521f1f833bee
Successfully built omnivoice
Looking in indexes: https://download.pytorch.org/whl/cu118


In [ ]:
!omnivoice-demo --share

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
2026-08-18 07:32:10,455 root INFO: Loading model from k2-fsa/OmniVoice, device=cuda ...
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0% 0/13 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/806M [00:00<?, ?B/s]          Warning: You are sending unauthenticated requests to the HF

In [ ]:
!rm -rf OmniVoice-Colab

In [ ]:
import os
import psutil
import gc
try:
  import torch
  HAS_TORCH = True
except ImportError:
  HAS_TORCH = False

def get_server_health_report():
  print("="*50)
  print("🔍 BÁO CÁO TÌNH TRẠNG MÁY CHỦ COLAB")
  print("="*50)

  # 1. CPU
  cpu_cores = os.cpu_count()
  print(f"🖥️ CPU: {cpu_cores} cores logic")

  # 2. RAM Hệ thống
  ram = psutil.virtual_memory()
  total_ram_gb = ram.total / (1024**3)
  available_ram_gb = ram.available / (1024**3)
  used_ram_gb = total_ram_gb - available_ram_gb
  print(f"🧠 RAM: Tổng {total_ram_gb:.1f} GB | Đã dùng {used_ram_gb:.1f} GB | Trống {available_ram_gb:.1f} GB")

  # 3. GPU VRAM
  total_vram_gb = 0
  available_vram_gb = 0

  if HAS_TORCH and torch.cuda.is_available():
      gpu_name = torch.cuda.get_device_name(0)

      # Ép dọn rác bộ nhớ để đo lường chính xác nhất
      gc.collect()
      torch.cuda.empty_cache()

      free_vram, total_vram = torch.cuda.mem_get_info(0)
      total_vram_gb = total_vram / (1024**3)
      available_vram_gb = free_vram / (1024**3)
      used_vram_gb = total_vram_gb - available_vram_gb

      print(f"🎮 GPU: {gpu_name}")
      print(f"💾 VRAM: Tổng {total_vram_gb:.1f} GB | Đã dùng {used_vram_gb:.1f} GB | Trống {available_vram_gb:.1f} GB")
  else:
      print(f"🎮 GPU: KHÔNG TÌM THẤY GPU (Colab đang chạy bằng CPU)")

  return {
      "cpu_cores": cpu_cores,
      "available_ram_gb": available_ram_gb,
      "available_vram_gb": available_vram_gb,
      "total_vram_gb": total_vram_gb
  }

def calculate_max_workers(health_data, model_vram_gb=4.0, is_shared_model=True):
  print("\n" + "="*50)
  print("⚙️ TÍNH TOÁN SỐ LUỒNG (MAX_WORKERS) AN TOÀN")
  print("="*50)

  # Biến số an toàn để Colab không bị sập OOM (Hết bộ nhớ)
  SAFE_MARGIN_VRAM = 2.0  # Cần chừa 2GB VRAM để chứa các tensor sinh ra lúc xử lý âm thanh
  SAFE_MARGIN_RAM = 2.0   # Chừa 2GB RAM cho hệ điều hành

  # Giới hạn theo CPU và RAM
  max_by_cpu = health_data["cpu_cores"]
  # Giả định 1 worker xử lý tốn thêm khoảng 1.5GB RAM hệ thống
  max_by_ram = max(1, int((health_data["available_ram_gb"] - SAFE_MARGIN_RAM) / 1.5))

  max_workers = 1 # Mặc định thấp nhất

  if health_data["total_vram_gb"] > 0:
      if is_shared_model:
          max_workers = min(max_by_cpu, max_by_ram)
          print("Cơ chế: Dùng chung 1 model (Đề xuất trên Colab)")
          print("Phân tích: VRAM chỉ chứa 1 bản model. Nút thắt là CPU và RAM hệ thống.")
      else:
          usable_vram = health_data["available_vram_gb"] - SAFE_MARGIN_VRAM
          max_by_vram = max(1, int(usable_vram / model_vram_gb))
          max_workers = min(max_by_cpu, max_by_ram, max_by_vram)
          print("Cơ chế: Mỗi luồng tự tạo lại model (Rất tốn VRAM)")
          print(f"Phân tích: GPU VRAM trống cho phép tối đa {max_by_vram} luồng (ước tính mỗi luồng {model_vram_gb}GB).")
  else:
      max_workers = min(max_by_cpu, max_by_ram)
      print("Cảnh báo: Bạn đang không dùng GPU. Tốc độ tạo âm thanh sẽ cực kỳ chậm.")

  print(f"\n=> 🚀 MAX_WORKERS ĐỀ XUẤT: {max_workers}")
  return max_workers

# ===== CHẠY THỬ =====
print("--- ĐANG KIỂM TRA MÁY CHỦ COLAB ---")
current_health = get_server_health_report()

# Tự động tính số luồng an toàn.
# CHÚ Ý: Nếu code của bạn load model mỗi worker 1 lần, đổi is_shared_model=False
# Nếu model OmniVoice của bạn nặng hơn 4GB, hãy sửa model_vram_gb=5.0 hoặc cao hơn.
my_max_workers = calculate_max_workers(current_health, model_vram_gb=4.0, is_shared_model=True)

--- ĐANG KIỂM TRA MÁY CHỦ COLAB ---
🔍 BÁO CÁO TÌNH TRẠNG MÁY CHỦ COLAB
🖥️ CPU: 2 cores logic
🧠 RAM: Tổng 12.7 GB | Đã dùng 1.6 GB | Trống 11.1 GB
🎮 GPU: Tesla T4
💾 VRAM: Tổng 14.6 GB | Đã dùng 0.1 GB | Trống 14.5 GB

⚙️ TÍNH TOÁN SỐ LUỒNG (MAX_WORKERS) AN TOÀN
Cơ chế: Dùng chung 1 model (Đề xuất trên Colab)
Phân tích: VRAM chỉ chứa 1 bản model. Nút thắt là CPU và RAM hệ thống.

=> 🚀 MAX_WORKERS ĐỀ XUẤT: 2
